# SR-KV Phase 6 - hyperparameter sweep + 3B transfer

No re-sweep on 3B: the question is whether the 1.5B config transfers.

Generated by `scripts/kaggle_kernel.py`. Thin by design: it clones the repo and
calls Makefile targets, so what runs here is exactly what runs locally.


In [ ]:
!nvidia-smi || echo 'no GPU (expected for phase 7)'
import torch
print('cuda:', torch.cuda.is_available())


In [ ]:
import subprocess, sys

REPO = 'https://github.com/Dhruvp18/FYP-SR_KV.git'
WORKDIR = '/kaggle/working/sr-kv'

def sh(cmd, cwd=WORKDIR, check=True):
    """Run a shell command, streaming output; raise so a failure fails the kernel."""
    print('+', cmd, flush=True)
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and result.returncode != 0:
        raise SystemExit(f'FAILED ({result.returncode}): {cmd}')
    return result.returncode


In [ ]:
import os
if os.path.isdir(WORKDIR):
    sh('git pull -q', check=False)
else:
    sh(f'git clone -q {REPO} {WORKDIR}', cwd='/kaggle/working')
sh('git log --oneline -1')


In [ ]:
# transformers 5.x is required (src/compat.py raises otherwise). torch ships with the image.
sh("pip install -q -U 'transformers>=5.0' accelerate bitsandbytes", cwd='/kaggle/working')
import transformers; print('transformers', transformers.__version__)


## Restore results from earlier phases

Any kernel listed in `kernel_sources` is mounted under `/kaggle/input`. Copying its
`.jsonl` files in is what makes this run resume instead of redoing finished work.


In [ ]:
import glob, shutil, os
os.makedirs(f'{WORKDIR}/results', exist_ok=True)
restored = 0
for pattern in ('/kaggle/input/*/results/*.jsonl', '/kaggle/input/*/sr-kv/results/*.jsonl'):
    for src in glob.glob(pattern):
        shutil.copy(src, f'{WORKDIR}/results/')
        restored += 1
print(f'restored {restored} result file(s)')
sh('ls -la results | head -20', check=False)


## Quick CPU test suite

Two minutes, no GPU, no downloads. Cheapest possible way to catch a broken commit
before spending quota on it.


In [ ]:
sh('python -m pytest -q')


## Run phase 6

Resumable: every finished task is fsynced to `results/*.jsonl`, so if this session is
killed, re-running this same kernel continues from where it stopped.


In [ ]:
sh('make phase6-sweep MODEL=qwen2.5-1.5b MODEL3B=llama3.2-3b BUDGET=0.3 SAMPLES=3 SHARD=0 NSHARDS=1')
sh('make phase6-3b MODEL=qwen2.5-1.5b MODEL3B=llama3.2-3b BUDGET=0.3 SAMPLES=3 SHARD=0 NSHARDS=1')
sh('make gate6 MODEL=qwen2.5-1.5b MODEL3B=llama3.2-3b BUDGET=0.3 SAMPLES=3 SHARD=0 NSHARDS=1')


## Results are the kernel output

Everything under `/kaggle/working` becomes this kernel's output, so `results/` and
`figures/` are pulled down by `kaggle_kernel.py pull` and mounted by the next phase.


In [ ]:
sh('ls -la results figures 2>/dev/null | head -40', check=False)
